## Difference-in-Differences Estimation

This notebook implements baseline Difference-in-Differences and event-study
specifications using the monthly HS8-level panel constructed in earlier steps.

### Estimation workflow (pseudocode)

- Load `did_panel_monthly.csv`
- Define panel unit: `product = hs8`
- Filter sample:
  - non-missing outcome
  - non-missing `post`, `highgap`, and `date`

For each outcome in:
- `ln_customs_value`
- `ln_first_unit_qty`
- `ln_unit_value`

Estimate the following models:

- **m1:**  
  `outcome ~ highgap × post`

- **m2:**  
  `outcome ~ highgap × post | product`

- **m3:**  
  `outcome ~ highgap × post | date`

- **m4:**  
  `outcome ~ highgap × post | product + date`

- Cluster standard errors by `product`
- Save results in a ladder (incremental FE) table

### Event-study specification

- Estimate:

      outcome ~ i(event_time, highgap, ref = -1) | product + date

- Cluster standard errors by `product`
- Save event-study plot with x-axis labeled in calendar years


In [5]:
import pandas as pd

df = pd.read_csv("../transformed_data/did_panel_monthly.csv")
df.shape

(824352, 17)

In [6]:
df_clean = df.dropna(
    subset=["ln_customs_value", "ln_first_unit_qty", "ln_unit_value", "post", "highgap", "date"]
)

df_clean.shape

(452504, 17)

In [13]:
from pyfixest.estimation import feols
from pyfixest.report import etable

# m1: no fixed effects
m1 = feols(
    "ln_customs_value ~ highgap * post",
    data=df_clean,
    vcov={"CRV1": "hs8"}
)

# m2: product fixed effects
m2 = feols(
    "ln_customs_value ~ highgap * post | hs8",
    data=df_clean,
    vcov={"CRV1": "hs8"}
)

# m3: date fixed effects
m3 = feols(
    "ln_customs_value ~ highgap * post | date",
    data=df_clean,
    vcov={"CRV1": "hs8"}
)

# m4: product + date fixed effects
m4 = feols(
    "ln_customs_value ~ highgap * post | hs8 + date",
    data=df_clean,
    vcov={"CRV1": "hs8"}
)


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 532 singleton fixed effect(s) detected. These observations are dropped from the model.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyfixest/estimation/feols_.py:2847: UserWarning: 
            1 variables dropped due to multicollinearity.
            The following variables are dropped: ['highgap'].
            
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyfixest/estimation/feols_.py:2847: UserWarning: 
            1 variables dropped due to multicollinearity.
            The following variables are dropped: ['post'].
            
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 532 singleton fixed effect(s) detected. These observati

In [67]:
table = etable([m1, m2, m3, m4])

with open("../results/did_ladder_customs.txt", "w") as f:
    f.write(table)

TypeError: write() argument must be str, not GT

In [68]:
table.to_markdown()

AttributeError: 'GT' object has no attribute 'to_markdown'

In [70]:
# 1. Generate your table as usual
table = etable([m1, m2, m3, m4])

# 2. 'table.data' extracts the Pandas DataFrame from the GT object
# Then we use 'to_string' to make it text
table_text = table.data.to_string(index=False)

# 3. Write to your file
with open("../results/did_ladder_customs.txt", "w") as f:
    f.write(table_text)

AttributeError: 'GT' object has no attribute 'data'

In [78]:
from stargazer.stargazer import Stargazer

# If stargazer supports your model object directly:
stargazer = Stargazer([m1, m2, m3, m4])

# Save as clean text
with open("../results/did_ladder_customs.txt", "w") as f:
    f.write(stargazer.render_html()) # Or .render_latex()

NotImplementedError: <class 'pyfixest.estimation.feols_.Feols'>

In [80]:
# 1. Create the object
table = etable([m1, m2, m3, m4])

# 2. Force access to the internal data frame
# We use ._data (the internal storage) to skip the GT wrapper
if hasattr(table, '_data'):
    raw_df = table._data
elif hasattr(table, 'data'):
    raw_df = table.data
else:
    # If all else fails, this is the most nuclear option
    raw_df = table

# 3. Write to .txt
with open("../results/did_ladder_customs.txt", "w") as f:
    f.write(raw_df.to_string())

AttributeError: 'GT' object has no attribute 'to_string'

In [72]:
%pip install stargazer


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [69]:
from tabulate import tabulate

# 1. Get the GT object
table = etable([m1, m2, m3, m4])

# 2. Access the underlying dataframe (.data) and format as text
# 'simple' or 'github' formats look best in .txt files
text_output = tabulate(table.data, tablefmt="simple", headers="keys")

with open("../results/did_ladder_customs.txt", "w") as f:
    f.write(text_output)

AttributeError: 'GT' object has no attribute 'data'

In [43]:
import pandas as pd
from pyfixest.report import etable

models = [m1, m2, m3, m4]
model_names = ["m1_customs", "m2_customs", "m3_customs", "m4_customs"]
variables = ["Intercept", "highgap", "post", "highgap:post"]

# Hard-code fixed effects for each model
fe_list = [
    [],                # m1: no fixed effects
    ["hs8"],           # m2: product
    ["date"],          # m3: date
    ["hs8", "date"]    # m4: product + date
]

# Function for significance stars
def stars(coef, se):
    if se == 0 or se is None:
        return ""
    t = abs(coef / se)
    if t >= 3.29: return '***'
    elif t >= 2.58: return '**'
    elif t >= 1.96: return '*'
    else: return ''

# Collect coefficients + SEs
table_data = {}
for name, model in zip(model_names, models):
    coefs = model.coef()   # pyfixest coef
    ses   = model.se()     # pyfixest se
    col = []
    for var in variables:
        if var in coefs:
            col.append(f"{coefs[var]:.4f}{stars(coefs[var], ses[var])} ({ses[var]:.4f})")
        else:
            col.append("")
    table_data[name] = col

coef_df = pd.DataFrame(table_data, index=variables)

# Fixed effects
fe_rows = ["product FE", "date FE"]
fe_data = {}
for name, fe_vars in zip(model_names, fe_list):
    fe_data[name] = [
        "Yes" if "hs8" in fe_vars else "No",
        "Yes" if "date" in fe_vars else "No",
    ]

fe_df = pd.DataFrame(fe_data, index=fe_rows)

# Stats: Observations, R2, Within R2
stats_df = pd.DataFrame({
    name: [
        f"{model._N:,}",                      # observations
        f"{model._r2:.5f}",                   # R2
        f"{model._r2_within:.5f}"             # Within R2
    ]
    for name, model in zip(model_names, models)
}, index=["Observations", "R2", "Within R2"])

# Combine
full_table = pd.concat([coef_df, fe_df, stats_df])

# Write to text file
with open("../results/did_ladder_customs.txt", "w") as f:
    f.write(full_table.to_string())

In [87]:
import pandas as pd
from linearmodels.panel import PanelOLS, compare

# 1. Convert 'date' to datetime objects so linearmodels is happy
df_clean['date'] = pd.to_datetime(df_clean['date'])

# 2. Set the MultiIndex [Entity, Time]
df_panel = df_clean.set_index(['hs8', 'date'])

# 3. Define and fit the models
# Note: Added 'cov_type="clustered", cluster_entity=True' to match your vcov={"CRV1": "hs8"}
# Model 1: No FE - Standard OLS
m1 = PanelOLS.from_formula("ln_customs_value ~ 1 + highgap * post", 
                           data=df_panel).fit(cov_type="clustered", cluster_entity=True)

# Model 2: Product FE - highgap is absorbed (it's constant per hs8)
m2 = PanelOLS.from_formula("ln_customs_value ~ post + highgap:post + EntityEffects", 
                           data=df_panel, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

# Model 3: Date FE - post is absorbed (it's constant per date)
m3 = PanelOLS.from_formula("ln_customs_value ~ highgap + highgap:post + TimeEffects", 
                           data=df_panel, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

# Model 4: Both FE - highgap and post are both absorbed
m4 = PanelOLS.from_formula("ln_customs_value ~ highgap:post + EntityEffects + TimeEffects", 
                           data=df_panel, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

# Generate and save the TXT table
# Use 'stat' instead of 'statistics'
res_table = compare(
    {"m1_customs": m1, "m2_customs": m2, "m3_customs": m3, "m4_customs": m4}, 
    stars=True,
    precision='std_errors'  # This puts Standard Errors in the parentheses
)

# Convert to text and swap the labels to match your original preference
table_text = res_table.summary.as_text()
table_text = table_text.replace("Entity", "product").replace("Time", "date")

# Save to your file
with open("../results/did_ladder_customs.txt", "w") as f:
    f.write(table_text)

print("Table successfully exported with Standard Errors!")

/var/folders/hq/hfb1chlj2pxg9shnwqj9t7jw0000gn/T/ipykernel_14212/2842675765.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['date'] = pd.to_datetime(df_clean['date'])


Table successfully exported with Standard Errors!
